# Day 8 — Boosting & Hyperparameter Tuning: Delivery Delays
This notebook reads the supplied dataset and tuning-result CSVs. It reports the saved results as recorded; it does not fabricate a metric name or a final test score.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('internB_delivery_delays.csv')
XGB_RESULTS_PATH = Path('xgb_randomized_search_results.csv')
OPTUNA_PATH = Path('optuna_trials.csv')

for path in [DATA_PATH, XGB_RESULTS_PATH, OPTUNA_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Put the supplied CSV files next to this notebook.')

data = pd.read_csv(DATA_PATH)
xgb_results = pd.read_csv(XGB_RESULTS_PATH)
optuna_trials = pd.read_csv(OPTUNA_PATH)
print('Dataset shape:', data.shape)
display(data.head())

## 1. Dataset checks
Target is `is_delayed`. Inspect imbalance, missingness, and potentially suspicious numeric values before fitting any model.

In [ ]:
print('Target counts:')
display(data['is_delayed'].value_counts(dropna=False).rename_axis('is_delayed').to_frame('count'))
print('Missing values:')
display(data.isna().sum().sort_values(ascending=False).to_frame('missing_count'))
print('Numeric summary:')
display(data.select_dtypes(include='number').describe().T)

## 2. Randomized-search results
The result CSV contains mean and standard deviation of cross-validation scores, five fold scores, and parameter settings. The metric is not named in this file alone.

In [ ]:
display(xgb_results.sort_values('rank_test_score').head(10))
best_xgb = xgb_results.loc[xgb_results['rank_test_score'].idxmin()]
print('Best rank:', best_xgb['rank_test_score'])
print('Mean CV score:', best_xgb['mean_test_score'])
print('CV score standard deviation:', best_xgb['std_test_score'])
print('Parameters:', best_xgb['params'])

## 3. Optuna trials
The highest `value` is the best recorded objective value, but its metric interpretation must be checked in the original Optuna objective code.

In [ ]:
display(optuna_trials.sort_values('value', ascending=False))
best_trial = optuna_trials.loc[optuna_trials['value'].idxmax()]
print('Best trial number:', best_trial['number'])
print('Best recorded value:', best_trial['value'])
display(best_trial.to_frame('best_trial').drop(labels=['datetime_start', 'datetime_complete', 'duration'], errors='ignore'))

## 4. Reporting checklist
- Verify which metric was optimized from the original training code.
- Confirm that randomized search and Optuna use comparable validation splits and objectives before comparing scores.
- Report the held-out test score only if it exists in the original experiment.
- Explain missing-value handling, target leakage safeguards, and unusual distance values.

## Results from supplied CSVs
- Best randomized-search mean CV score: approximately 0.5996.
- Highest recorded Optuna objective: approximately 0.6095 (trial 1).
These values are copied from the supplied result artifacts; the metric name is not inferred.